# Demo: generate ops-app demand gzip shard -> `raw_data/demand/`

Reads random rows from `/Volumes/.../raw_data/310.csv.gz` and writes a uniquely named demand shard into the Auto Loader incoming folder for `ops_app_bronze_building_hourly_demand`.

Output schema (headerless CSV):
1. `building_id`
2. `event_ts`
3. `demand_users`
4. `traffic_mix`
5. `indoor_penetration_factor`

In [ ]:
SOURCE_GZIP = "/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/310.csv.gz"
DEMAND_DIR = "/Volumes/cmegdemos_catalog/network_analytics_enablement/raw_data/demand"
CATALOG = "cmegdemos_catalog"
SCHEMA = "network_analytics_enablement"

MIN_SAMPLE_ROWS = 6_000
MAX_SAMPLE_ROWS = 30_000

In [ ]:
import os
import uuid
from datetime import datetime, timezone

import numpy as np
import pandas as pd

assert os.path.exists(SOURCE_GZIP), f"Missing source file: {SOURCE_GZIP}"
os.makedirs(DEMAND_DIR, exist_ok=True)

bldg_pdf = spark.sql(
    f"SELECT building_id, COALESCE(building_height, 20.0) AS building_height "
    f"FROM {CATALOG}.{SCHEMA}.gold_downtown_building_coverage"
).toPandas()
assert not bldg_pdf.empty, "No rows found in gold_downtown_building_coverage"

seed_df = pd.read_csv(SOURCE_GZIP, header=None, compression="gzip", dtype=str, low_memory=False)
seed_n = len(seed_df)
rng = np.random.default_rng()
hi = min(MAX_SAMPLE_ROWS, seed_n)
lo = min(MIN_SAMPLE_ROWS, hi)
target_n = int(rng.integers(lo, hi + 1)) if hi >= lo else seed_n
target_n = max(1, min(target_n, seed_n))
sampled_seed = seed_df.sample(n=target_n, replace=False, random_state=rng)

sampled_bldg = bldg_pdf.sample(n=target_n, replace=True, random_state=int(rng.integers(0, 2_000_000_000)))
height = pd.to_numeric(sampled_bldg["building_height"], errors="coerce").fillna(20.0)

now = datetime.now(timezone.utc)
hours_back = rng.integers(0, 14 * 24, size=target_n)
event_ts = pd.to_datetime(
    [
        (now - pd.Timedelta(hours=int(h))).replace(minute=0, second=0, microsecond=0)
        for h in hours_back
    ]
)
h = event_ts.dt.hour
dow = event_ts.dt.dayofweek

base = 8 + (height / 8.0)
hour_boost = np.where((h >= 7) & (h <= 10), 2.2, np.where((h >= 16) & (h <= 20), 2.6, np.where((h >= 0) & (h <= 5), 0.45, 1.0)))
weekend_factor = np.where(dow >= 5, 0.7, 1.0)
noise = 0.9 + rng.random(target_n) * 0.35
seed_influence = 0.95 + (pd.to_numeric(sampled_seed[10], errors="coerce").fillna(1).clip(0, 1) * 0.1)
demand_users = np.maximum(1, np.round(base * hour_boost * weekend_factor * noise * seed_influence)).astype(int)

traffic_mix = np.where((h >= 18) & (h <= 23), "video_heavy", np.where((h >= 7) & (h <= 9), "commute", "balanced"))
indoor_pen = np.round(0.45 + rng.random(target_n) * 0.5, 3)

out_pdf = pd.DataFrame(
    {
        "building_id": sampled_bldg["building_id"].astype("int64").values,
        "event_ts": event_ts.strftime("%Y-%m-%d %H:%M:%S"),
        "demand_users": demand_users,
        "traffic_mix": traffic_mix,
        "indoor_penetration_factor": indoor_pen,
    }
)

ts = now.strftime("%Y%m%dT%H%M%SZ")
suffix = uuid.uuid4().hex[:10]
out_name = f"ops_demand_shard_{ts}_{suffix}.csv.gz"
out_path = os.path.join(DEMAND_DIR, out_name)
out_pdf.to_csv(out_path, header=False, index=False, compression="gzip")

print(f"Seed rows sampled from 310.csv.gz: {target_n:,}")
print(f"Wrote demand shard: {out_path}")
print("Next: run ops_app_network_analytics_pipeline")